# Scinapse — explore a paper field as a layered graph

Prototype for the **Pre-GTC HCLS hackathon** (*Research Paper Discovery*). Given a few **seed papers**, it builds an interactive graph that shows *how the field evolved around them*: newest relevant work at the top, seeds at the bottom, at most **3 citation layers** in between — plus several ways to judge each paper at a glance.

**What the encoding means**

| Channel | Meaning |
|---|---|
| vertical position | **publication year** — newest papers at the top, seeds anchor the bottom |
| node color | **venue quality**: red = journal publishes a suspicious number of papers/year (paper-mill territory) → gray midpoint → green = normal specialist venue; **seeds are constant orange** so they're always findable |
| node size | composite relevance score (all nodes round) |
| edge color + pattern | **why it's relevant**: blue solid = *cites* (arrow points at what it cites) · amber dashed = *abstract similarity* · pink dotted = *shared author* |
| edge width | strength of the connection — hover any edge for the reason and the exact number |
| click a node | copies its DOI link to the clipboard |
| ⚠ red border | retracted publication |

The quality ramp is red↔green as requested; because that pair is the classic colorblind trap, the exact numeric value also appears in every tooltip and in the results table — color never carries the meaning alone.

**Pipeline**: resolve seeds → forward-citation BFS ×3 with pruning → top-entry (latest by author / by topic) → venue-quality enrichment → similarity + author links → relevance explanations → interactive graph + table.

**How to explore**: the graph starts deliberately sparse. Use the **exploration toolkit** at the bottom — `more()` to load additional candidates per layer on demand, `inspect("id")` for everything about a paper, `expand("id")` to re-grow the graph around a promising find, `reset_explore()` to start over.

## Setup

```bash
python -m pip install pandas numpy scikit-learn networkx pyvis openai
```

Data sources and their limits:

- **Semantic Scholar Graph API** (no key) — discovery: DOI resolution, citing papers, author feeds, topic search. It is a *shared* pool, so this notebook paces and retries every call and caches all responses under `.scinapse_cache/` — reruns are instant.
- **OpenAlex** (no key) — venue statistics for the quality heuristic. Only *single-record* endpoints are used; those are free of the daily credit pool that query/search endpoints consume on shared IPs.
- **Nebius Token Factory** (optional) — embeddings + LLM relevance explanations. Without a key the notebook falls back to TF-IDF similarity and evidence-based template explanations.

In [1]:
import html as _html
import json
import math
import os
import time
import hashlib
from collections import Counter, defaultdict
from getpass import getpass
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import IFrame, display
from pyvis.network import Network
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Theme -----------------------------------------------------------------
# Quality ramp: red (low) <-> green (high) with a gray midpoint, as requested.
# NOTE: red<->green is the classic color-vision-deficiency trap, so the value
# is never carried by color alone — every tooltip and the results table show
# the numeric quality, and the ramp's lightness is monotone within each arm.
# Edge identity is doubled with dash patterns + arrows (worst pair dE 41+).
THEME = "light"  # "light" | "dark"

LIGHT = {
    "surface": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
    "grid": "#e1e0d9",
    # quality ramp: low <------ midpoint ------> high
    "quality": ["#8f1f1f", "#d63b3b", "#e87f78", "#f0efec", "#5ec45e", "#1f9d1f", "#0b5d0b"],
    "seed": "#eb6834",          # seeds are constant orange — always findable
    "unknown": "#c3c2b7",       # no venue stats
    "citation": "#2a78d6", "semantic": "#eda100", "author": "#e87ba4",
    "retracted": "#d03b3b",
}
DARK = {
    "surface": "#1a1a19", "ink": "#ffffff", "ink2": "#c3c2b7", "muted": "#898781",
    "grid": "#2c2c2a",
    "quality": ["#9c2626", "#d64948", "#ef9d97", "#383835", "#1f7a1f", "#2fb62f", "#8ce08c"],
    "seed": "#d95926",
    "unknown": "#52514e",
    "citation": "#3987e5", "semantic": "#c98500", "author": "#d55181",
    "retracted": "#d03b3b",
}
T = LIGHT if THEME == "light" else DARK
print(f"theme: {THEME}")

theme: light


In [2]:
RESEARCH_QUESTION = "Which single-cell methods help study melanoma immune cells?"

SEED_DOIS = [
    "10.1126/science.aad0501",        # Dissecting the multicellular ecosystem of metastatic melanoma (Science 2016)
    "10.1016/j.cell.2018.10.038",     # T cell states associated with checkpoint-blockade response (Cell 2018)
    "10.1038/s41586-021-03704-y",     # Phenotype, specificity, avidity of antitumour CD8 T cells (Nature 2021)
    "10.1038/s41586-022-04682-5",     # Helper and regulatory antitumour CD4 T cells (Nature 2022)
    "10.1158/1078-0432.CCR-21-1694",  # Single-cell characterization of the melanoma immune microenvironment (CCR 2021)
]

# Graph shape — deliberately sparse by default; use more() to load extra
# candidates per layer on demand ("dynamically load additional connections")
N_LAYERS = 3            # citation layers above the seeds
TOP_K_PER_LAYER = 8     # candidates kept per layer — "only top relevant things"
TOP_ENTRY_N = 6         # brand-new papers pinned into the top layer
RECENT_MIN_YEAR = 2025  # what counts as "latest" for top-entry papers

# Quality heuristic: journal papers/year -> quality in [0, 1].
#   <=200/yr -> 1.0 ; ~1,000/yr -> ~0.55 ; ~3,000+/yr -> ~0.2 ; retracted -> 0.0
# It is a *venue* heuristic (paper mills publish thousands per year), not a
# verdict on the paper — the tooltip always shows the raw number.

CACHE_DIR = Path(".scinapse_cache")
CACHE_DIR.mkdir(exist_ok=True)
GRAPH_HTML = "scinapse_graph.html"

EXTRA_SEEDS = []  # papers added while exploring, via expand()

if os.environ.get("SCINAPSE_SMOKE"):  # tiny cheap run for CI-style checking
    SEED_DOIS = SEED_DOIS[:2]
    N_LAYERS, TOP_K_PER_LAYER, TOP_ENTRY_N = 1, 4, 2
    print("SCINAPSE_SMOKE: 2 seeds / 1 layer / small caps")
print(f"{len(SEED_DOIS)} seeds · {N_LAYERS} layers · top {TOP_K_PER_LAYER}/layer · {TOP_ENTRY_N} latest")

5 seeds · 3 layers · top 8/layer · 6 latest


In [3]:
LAST_CALL = {"t": 0.0}
MAILTO = "nonvisual.wrk@gmail.com"  # polite-pool identifier for both APIs
S2 = "https://api.semanticscholar.org/graph/v1"
OA = "https://api.openalex.org"


def _cache_key(*parts):
    h = hashlib.sha1()
    for p in parts:
        h.update(str(p).encode())
    return CACHE_DIR / (h.hexdigest()[:20] + ".json")


def http_json(url, *, headers=None, data=None, min_interval=1.25, retries=7, cache=True):
    """GET/POST JSON with disk cache, polite pacing and exponential backoff."""
    key = _cache_key(url, data)
    if cache and key.exists():
        return json.loads(key.read_text())
    hdrs = {"User-Agent": f"scinapse-prototype; mailto:{MAILTO}", "Accept": "application/json"}
    if headers:
        hdrs.update(headers)
    if data is not None:
        hdrs["Content-Type"] = "application/json"
    for attempt in range(retries):
        gap = min_interval - (time.time() - LAST_CALL["t"])
        if gap > 0:
            time.sleep(gap)
        LAST_CALL["t"] = time.time()
        try:
            req = Request(url, data=data, headers=hdrs, method="POST" if data is not None else "GET")
            with urlopen(req, timeout=60) as r:
                payload = json.load(r)
            if cache:
                key.write_text(json.dumps(payload))
            return payload
        except Exception as e:  # one backoff path for 429/5xx/timeouts
            code = getattr(e, "code", None)
            if code in (400, 403, 404, 422) or attempt == retries - 1:
                raise  # client errors are permanent — never retry them
            wait = min(2 ** attempt + 1, 30)
            print(f"    [{code or type(e).__name__}] retry {wait}s: {url[:80]}")
            time.sleep(wait)


def s2_get(path, _retries=None, **params):
    kw = {"min_interval": 1.25}
    if _retries:
        kw["retries"] = _retries
    return http_json(f"{S2}{path}?{urlencode(params)}", **kw)


def oa_get(path, **params):
    # single-record OpenAlex calls are free of the daily credit pool
    params.setdefault("mailto", MAILTO)
    return http_json(f"{OA}{path}?{urlencode(params)}", min_interval=0.15)


print("HTTP layer ready (cached, throttled, retrying)")

HTTP layer ready (cached, throttled, retrying)


In [4]:
TF_API_KEY = None            # or: getpass("Token Factory API key: ")
CHAT_MODEL = "nvidia/Nemotron-3_5-Lightning"
EMBED_MODEL = ""             # pick an embedding model from the Token Factory catalog

tf = None
if TF_API_KEY:
    from openai import OpenAI
    tf = OpenAI(base_url="https://api.tokenfactory.nebius.com/v1/", api_key=TF_API_KEY)
    print("Token Factory: on")
else:
    print("Token Factory: off -> TF-IDF similarity + evidence-template explanations")

Token Factory: off -> TF-IDF similarity + evidence-template explanations


In [5]:
S2_FIELDS = "paperId,title,year,venue,citationCount,abstract,authors,externalIds"

PAPERS = {}       # paperId -> record
EDGES = []        # {src, dst, type, strength(0..1, drives width), label, detail}
LAYER_POOLS = {}  # layer -> [(score, paperId, parent_ids, s2_record), ...] not yet shown


def short(pid):
    return pid[:8]


def short_title(t, n=34):
    t = (t or "").strip()
    return t if len(t) <= n else t[: n - 1] + "…"


def add_paper(p, *, layer=None, is_seed=False, entry=None):
    """Idempotently register a Semantic Scholar paper record."""
    pid = p.get("paperId")
    if not pid or not (p.get("title") or "").strip():
        return None
    rec = PAPERS.setdefault(pid, {
        "id": pid,
        "title": p["title"].strip(),
        "year": p.get("year"),
        "venue": p.get("venue") or "",
        "abstract": (p.get("abstract") or "").strip(),
        "authors": [(a["authorId"], a["name"]) for a in (p.get("authors") or []) if a.get("authorId")],
        "cites": p.get("citationCount") or 0,
        "doi": (p.get("externalIds") or {}).get("DOI"),
        "pmid": (p.get("externalIds") or {}).get("PubMed"),
        "is_seed": is_seed,
        "layer": layer,
        "entry": entry,          # "author" | "topic" for latest papers
        "cites_count": p.get("citationCount") or 0,
        "quality": None, "ppy": None, "source_name": None, "source_id": None,
        "retracted": False, "sem": None, "score": None, "evidence": [], "explanation": None,
    })
    if is_seed:
        rec["is_seed"] = True
    if layer is not None and (rec["layer"] is None or layer < rec["layer"]):
        rec["layer"] = layer
    return rec


def add_edge(src, dst, etype, strength, label="", detail=""):
    """strength in [0,1] drives edge width; detail is hover text."""
    if src == dst or src not in PAPERS or dst not in PAPERS:
        return False
    for e in EDGES:
        if e["src"] == src and e["dst"] == dst and e["type"] == etype:
            return False
    EDGES.append({"src": src, "dst": dst, "type": etype,
                  "strength": min(max(float(strength), 0.0), 1.0),
                  "label": label, "detail": detail})
    return True


def seed_author_ids():
    ids = set()
    for r in PAPERS.values():
        if r["is_seed"]:
            ids.update(a[0] for a in r["authors"])
    return ids


print("paper store ready")

paper store ready


In [6]:
def fetch_seeds(dois):
    for doi in dois:
        p = s2_get(f"/paper/DOI:{doi}", fields=S2_FIELDS)
        rec = add_paper(p, layer=0, is_seed=True)
        if rec:
            print(f"    L0 {rec['year']} cites={rec['cites']:>5} abs={'y' if rec['abstract'] else 'n'}  {short_title(rec['title'], 66)}")


def expand_layers(n_layers=N_LAYERS, top_k=TOP_K_PER_LAYER):
    """Forward-citation BFS: layer k cites layer k-1; newest work drifts upward.

    Semantic Scholar lists citers newest-first, so layer 1 fetches two windows
    per seed (recent frontier + older, established followers) — otherwise every
    layer-1 pick is from the last months and deeper layers have nothing newer
    left to build on. Deeper layers fetch only the recent window.
    """
    parents = [p for p, r in PAPERS.items() if r["is_seed"]]
    s_auth = seed_author_ids()
    for layer in range(1, n_layers + 1):
        # layer 1 additionally samples an older window (adaptive offset: a third
        # of the way into the parent's citer list) so the layer spans eras;
        # recency matters most for the top layer only.
        rec_w = {1: 0.3, 2: 0.6, 3: 1.0}.get(layer, 1.0)
        windows = [(0, 60)] if layer > 1 else [(0, 100), ("deep", 100)]
        hits, metas = defaultdict(list), {}
        for pid in parents:
            for offset, limit in windows:
                params = dict(fields=S2_FIELDS, limit=limit)
                if offset == "deep":
                    params["offset"] = max(150, min(1500, PAPERS[pid]["cites"] // 3))
                elif offset:
                    params["offset"] = offset
                try:
                    r = s2_get(f"/paper/{pid}/citations", **params)
                except Exception as e:
                    print(f"    ! citations for {short(pid)} failed: {e}")
                    continue
                for row in r.get("data") or []:
                    c = row.get("citingPaper") or {}
                    cid = c.get("paperId")
                    if not cid or not (c.get("title") or "").strip():
                        continue
                    hits[cid].append(pid)
                    if cid not in PAPERS:
                        metas[cid] = c
        fresh = []
        for cid, plist in hits.items():
            if cid in PAPERS:
                continue
            m = metas[cid]
            y = m.get("year") or 0
            a_ids = {x["authorId"] for x in (m.get("authors") or []) if x.get("authorId")}
            score = (
                2.0 * len(plist)
                + rec_w * (1.0 if y >= 2025 else 0.6 if y >= 2023 else 0.3 if y >= 2020 else 0.0)
                + 0.5 * math.log10(1 + (m.get("citationCount") or 0)) / 3.0
                + 0.8 * float(bool(a_ids & s_auth))
            )
            fresh.append((score, cid, plist))
        fresh.sort(reverse=True)
        # year-diversity quota: at most top_k//3 picks per publication year,
        # then fill any leftover slots by raw score
        kept, kept_ids, year_count = [], set(), Counter()
        cap = max(3, top_k // 3)
        for s, cid, plist in fresh:
            if len(kept) >= top_k:
                break
            y = (metas[cid].get("year") or 0)
            if year_count[y] >= cap:
                continue
            kept.append((s, cid, plist))
            kept_ids.add(cid)
            year_count[y] += 1
        for s, cid, plist in fresh:
            if len(kept) >= top_k:
                break
            if cid not in kept_ids:
                kept.append((s, cid, plist))
                kept_ids.add(cid)
        for _, cid, _ in kept:
            add_paper(metas[cid], layer=layer)
        for cid, plist in hits.items():          # record citation edges for all kept nodes
            if cid in PAPERS:
                for p in plist:
                    add_edge(cid, p, "citation", 0.55)
        LAYER_POOLS[layer] = [(s, cid, plist, metas[cid])
                              for s, cid, plist in fresh if cid not in kept_ids]
        if not kept:
            print(f"    layer {layer}: nothing new — stopping")
            break
        ys = sorted(PAPERS[cid]["year"] or 0 for _, cid, _ in kept)
        print(f"    layer {layer}: {len(hits)} candidates citing {len(parents)} parents -> kept {len(kept)} of top, pool {len(LAYER_POOLS[layer])} (years {ys[0]}..{ys[-1]})")
        parents = [cid for _, cid, _ in kept]


def add_top_entry(n=TOP_ENTRY_N):
    """Pin the newest relevant papers into the top layer.

    Two entry modes: newest works by authors that appear across the seeds,
    and a topic search for the research question. These papers are often too
    new to have citations — they anchor the 'latest' frontier.
    """
    pool, seen = [], set()
    counts = Counter(a[0] for r in PAPERS.values() if r["is_seed"] for a in r["authors"])
    anchors = [aid for aid, c in counts.most_common() if c >= 2][:3]
    if not anchors:
        anchors = [r["authors"][0][0] for r in PAPERS.values() if r["is_seed"] and r["authors"]][:3]
    for aid in anchors:
        try:
            r = s2_get(f"/author/{aid}/papers", fields=S2_FIELDS, limit=25)
        except Exception as e:
            print(f"    ! author feed {aid}: {e}")
            continue
        for p in (r.get("data") or r.get("papers") or []):   # shape: {offset, next, data}
            pid = p.get("paperId")
            if pid and pid not in PAPERS and pid not in seen and (p.get("year") or 0) >= RECENT_MIN_YEAR:
                seen.add(pid)
                pool.append(("author", p))
    try:
        r = s2_get("/paper/search", _retries=4, query=RESEARCH_QUESTION,
                   year=f"{RECENT_MIN_YEAR}-{time.gmtime().tm_year}",
                   fields=S2_FIELDS, limit=25)
        for p in r.get("data") or []:
            pid = p.get("paperId")
            if pid and pid not in PAPERS and pid not in seen:
                seen.add(pid)
                pool.append(("topic", p))
    except Exception as e:
        print(f"    ! topic search: {e}")
    pool.sort(key=lambda ip: ((ip[1].get("year") or 0), ip[1].get("citationCount") or 0), reverse=True)
    for entry, p in pool[:n]:
        rec = add_paper(p, layer=N_LAYERS, entry=entry)
        if rec:
            print(f"    latest[{entry:>6}] {rec['year']} cites={rec['cites']:>3}  {short_title(rec['title'], 62)}")


print("discovery functions ready")

discovery functions ready


In [7]:
PREPRINT_HINTS = ("biorxiv", "medrxiv", "arxiv", "ssrn", "chemrxiv", "research square", "preprint")


def venue_quality(ppy, retracted=False, h_index=0, stype="journal", name=""):
    """Journal papers/year -> soft quality in [0,1]; None = unknown venue.

    - retracted -> 0.0
    - preprint servers / repositories -> None (not peer-review-graded venues)
    - flagship weeklies (h-index >= 900) also publish thousands per year -> floor 0.75
    - otherwise: <=200/yr -> 1.0 ... ~3,200+/yr -> ~0.2 (paper-mill territory)
    """
    if retracted:
        return 0.0
    lname = (name or "").lower()
    if "preprint" in (stype or "") or "repository" in (stype or "") or any(h in lname for h in PREPRINT_HINTS):
        return None
    if not ppy:
        return None
    l = math.log10(max(ppy, 1))
    if l <= 2.3:                                   # <=200 papers/yr
        q = 1.0
    elif l <= 3.0:                                 # 200..1,000
        q = 1.0 - 0.45 * (l - 2.3) / 0.7
    elif l <= 3.5:                                 # 1,000..~3,200
        q = 0.55 - 0.30 * (l - 3.0) / 0.5
    else:                                          # paper-mill territory
        q = max(0.10, 0.25 - 0.15 * (l - 3.5))
    if (h_index or 0) >= 900:
        q = max(q, 0.75)
    return q


def enrich_quality():
    """OpenAlex single-record calls (free of the daily credit pool):
    DOI -> work (retraction, journal) -> source (papers/year)."""
    src_ids = {}
    matched = 0
    for rec in PAPERS.values():
        if not rec["doi"]:
            continue
        try:
            w = oa_get(f"/works/doi:{rec['doi']}", select="is_retracted,primary_location")
        except Exception:
            continue
        matched += 1
        rec["retracted"] = bool(w.get("is_retracted"))
        src = ((w.get("primary_location") or {}).get("source") or {})
        sid = (src.get("id") or "").rsplit("/", 1)[-1]
        if sid:
            rec["source_id"] = sid
            rec["source_name"] = src.get("display_name") or rec["venue"] or None
            src_ids.setdefault(sid, None)
    for sid in list(src_ids):
        try:
            s = oa_get(f"/sources/{sid}")  # small record; no select (h_index sits in summary_stats)
        except Exception:
            continue
        cby = sorted(s.get("counts_by_year") or [], key=lambda c: -c.get("year", 0))
        ppy = next((c["works_count"] for c in cby if c.get("year") == 2025), None)  # last full year
        if ppy is None and cby:
            ppy = cby[0].get("works_count")
        src_ids[sid] = {"name": s.get("display_name"), "ppy": ppy,
                        "h": (s.get("summary_stats") or {}).get("h_index") or 0,
                        "type": s.get("type") or "journal"}
    for rec in PAPERS.values():
        info = src_ids.get(rec["source_id"])
        rec["ppy"] = info["ppy"] if info else None
        rec["source_name"] = (info["name"] if info else None) or rec["source_name"] or rec["venue"]
        rec["quality"] = venue_quality(rec["ppy"], rec["retracted"],
                                       h_index=info["h"] if info else 0,
                                       stype=info["type"] if info else "journal",
                                       name=rec["source_name"] or "")
    print(f"    matched {matched} DOIs on OpenAlex; {len(src_ids)} distinct journals")
    biggest = sorted({(r["ppy"], r["source_name"] or "?") for r in PAPERS.values() if r["ppy"]}, reverse=True)[:8]
    for ppy, name in biggest:
        print(f"      {ppy:>7,} papers/yr  {name}")
    n_unk = sum(1 for r in PAPERS.values() if r["quality"] is None)
    if n_unk:
        print(f"      {n_unk} papers without venue stats (gray nodes)")


print("quality functions ready")

quality functions ready


In [8]:
def build_semantic():
    """Abstract/title similarity: TF-IDF locally, Token Factory embeddings if configured."""
    ids = list(PAPERS)
    docs = [f"{PAPERS[p]['title']}. {PAPERS[p]['abstract']}".strip()[:4000] for p in ids]
    used, M = "tfidf", None
    if tf is not None and EMBED_MODEL:
        try:
            vecs = []
            for i in range(0, len(docs), 32):
                resp = tf.embeddings.create(model=EMBED_MODEL, input=docs[i:i + 32])
                vecs.extend(d.embedding for d in resp.data)
            M = cosine_similarity(np.asarray(vecs, dtype=float))
            used = "embeddings"
        except Exception as e:
            print(f"    embeddings unavailable ({e}) -> TF-IDF fallback")
            M = None
    if M is None:
        tfv = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1, sublinear_tf=True)
        M = cosine_similarity(tfv.fit_transform(docs))
    threshold = 0.55 if used == "embeddings" else 0.15  # TF-IDF cosine runs lower than embeddings
    n_abs = sum(1 for p in ids if PAPERS[p]["abstract"])
    seed_idx = [i for i, p in enumerate(ids) if PAPERS[p]["is_seed"]]
    for i, pid in enumerate(ids):
        rec = PAPERS[pid]
        if rec["is_seed"]:
            continue
        if seed_idx:
            rec["sem"] = round(float(max(M[i][j] for j in seed_idx)), 3)
        added = 0
        for j in sorted((j for j in range(len(ids)) if j != i), key=lambda j: -M[i][j]):
            if added >= 2 or M[i][j] < threshold:
                break
            sim = round(float(M[i][j]), 3)
            if add_edge(pid, ids[j], "semantic", min(sim * 2, 1.0),
                        detail=f"cosine {sim:.2f} ({used})"):
                added += 1
    print(f"    similarity via {used} (threshold {threshold}); {n_abs}/{len(ids)} papers have abstracts")


def build_author_edges():
    """Shared-author edges, kept only when they touch a seed or a latest-entry
    node (that is the relevance question), max 2 per node to limit clutter."""
    by_author = defaultdict(list)
    for pid, r in PAPERS.items():
        for aid, _name in r["authors"]:
            by_author[aid].append(pid)
    pairs, names = Counter(), {}
    for aid, pids in by_author.items():
        if len(pids) < 2:
            continue
        for i in range(len(pids)):
            for j in range(i + 1, len(pids)):
                a, b = sorted((pids[i], pids[j]))
                pairs[(a, b)] += 1
                names[(a, b)] = next(n for i2, n in PAPERS[a]["authors"] if i2 == aid)
    deg, added = Counter(), 0
    for (a, b), n in pairs.most_common():
        ra, rb = PAPERS[a], PAPERS[b]
        if not (ra["is_seed"] or rb["is_seed"] or ra["entry"] or rb["entry"]):
            continue
        if deg[a] >= 2 or deg[b] >= 2:
            continue
        if add_edge(a, b, "author", min(n / 3.0, 1.0), names[(a, b)],
                    detail=f"{n} shared author(s)"):
            deg[a] += 1
            deg[b] += 1
            added += 1
    print(f"    author edges: {added}")


print("similarity + author functions ready")

similarity + author functions ready


In [9]:
def _recency(y):
    return 1.0 if (y or 0) >= 2025 else 0.7 if (y or 0) >= 2023 else 0.4


def finalize():
    """Composite relevance score + human-readable evidence lines per paper."""
    seed_ids = {p for p, r in PAPERS.items() if r["is_seed"]}
    for rec in PAPERS.values():
        if rec["is_seed"]:
            rec["score"] = None
            rec["evidence"] = ["seed paper — your starting collection"]
            continue
        cites_seed = [e for e in EDGES if e["src"] == rec["id"] and e["type"] == "citation" and e["dst"] in seed_ids]
        cites_other = [e for e in EDGES if e["src"] == rec["id"] and e["type"] == "citation" and e["dst"] not in seed_ids]
        auth_e = [e for e in EDGES if e["type"] == "author" and rec["id"] in (e["src"], e["dst"])]
        rec["score"] = round(100 * (
            0.45 * min((rec["sem"] or 0) / 0.5, 1.0)
            + 0.25 * min(len(cites_seed) / 2.0, 1.0)
            + 0.15 * min(len(cites_other) / 3.0, 1.0)
            + 0.10 * _recency(rec["year"])
            + 0.05 * min(len(auth_e) / 2.0, 1.0)
        ))
        ev = []
        if cites_seed:
            names = "; ".join(short_title(PAPERS[e["dst"]]["title"], 40) for e in cites_seed[:3])
            ev.append(f"cites {len(cites_seed)} seed paper(s): {names}")
        if cites_other:
            ev.append(f"cites {len(cites_other)} discovered paper(s) on paths to the seeds")
        sem_e = [e for e in EDGES if e["type"] == "semantic" and rec["id"] in (e["src"], e["dst"])]
        if sem_e:
            best = max(sem_e, key=lambda e: e["strength"])
            other = best["dst"] if best["src"] == rec["id"] else best["src"]
            ev.append(f"abstract similarity {best.get('detail', '')} with {short_title(PAPERS[other]['title'], 40)}")
        if auth_e:
            ev.append("shares author(s): " + ", ".join(e["label"] for e in auth_e[:3]))
        if rec["entry"]:
            ev.append(f"latest work (entered by {rec['entry']}) — no citation links yet")
        rec["evidence"] = ev or ["no explicit connection extracted"]


def explain_all(limit=32):
    """Token Factory explanations (cached). Without a key: evidence templates."""
    targets = sorted((r for r in PAPERS.values() if not r["is_seed"]),
                     key=lambda r: -(r["score"] or 0))[:limit]
    for rec in targets:
        template = " | ".join(rec["evidence"])
        rec["explanation"] = template
        if tf is None:
            continue
        prompt = (
            f"Research question: {RESEARCH_QUESTION}\n\n"
            f"Paper: {rec['title']} ({rec['year']}, {rec['venue'] or 'unknown venue'})\n"
            f"Abstract: {rec['abstract'] or 'NOT AVAILABLE'}"[:1400] + "\n\n"
            f"Connections found automatically: {template}\n\n"
            "In at most two sentences, say why this paper may or may not matter for the "
            "research question, naming the connection type (cites-seed / similar-topic / "
            "same-author / latest-by-author). Use only the information above. End with a "
            "line 'Evidence: <short phrase quoted from the abstract, or none available>'."
        )
        key = _cache_key("tf-explain", CHAT_MODEL, prompt)
        suffix = key.with_suffix(".txt")
        if suffix.exists():
            rec["explanation"] = suffix.read_text()
            continue
        try:
            reply = tf.chat.completions.create(model=CHAT_MODEL, max_tokens=280,
                                               messages=[{"role": "user", "content": prompt}])
            rec["explanation"] = reply.choices[0].message.content.strip()
            suffix.write_text(rec["explanation"])
        except Exception as e:
            print(f"    ! explanation for {short(rec['id'])}: {e}")
    mode = "Token Factory" if tf is not None else "evidence templates"
    print(f"    explanations for top {min(limit, len(targets))} papers via {mode} (cached)")


print("scoring + explanation functions ready")

scoring + explanation functions ready


In [10]:
def run_pipeline(seed_dois=None):
    """Full rebuild around the seed set (cached calls make reruns cheap)."""
    t0 = time.time()
    global PAPERS, EDGES, LAYER_POOLS
    PAPERS, EDGES, LAYER_POOLS = {}, [], {}
    dois = list(seed_dois or SEED_DOIS) + list(EXTRA_SEEDS)
    print(f"1) seeds ({len(dois)})")
    fetch_seeds(dois)
    print("2) forward-citation layers")
    expand_layers()
    print("3) top entry: latest by author / by topic")
    add_top_entry()
    print("4) venue-quality enrichment (OpenAlex single-record calls)")
    enrich_quality()
    print("5) similarity, author links, scores, explanations")
    build_semantic()
    build_author_edges()
    finalize()
    explain_all()
    print(f"   -> {len(PAPERS)} nodes / {len(EDGES)} edges in {time.time() - t0:.0f}s")


print("pipeline ready")

pipeline ready


## Run the pipeline

Everything is cached in `.scinapse_cache/` — the first run talks to the APIs (a couple of minutes, mostly Semantic Scholar pacing), later runs are near-instant.

In [11]:
run_pipeline()

1) seeds (5)
    L0 2016 cites= 4550 abs=y  Dissecting the multicellular ecosystem of metastatic melanoma by …
    L0 2018 cites= 1930 abs=y  Defining T cell states associated with response to checkpoint imm…
    L0 2021 cites=  490 abs=y  Phenotype, specificity and avidity of antitumour CD8+ T cells in …
    L0 2022 cites=  175 abs=y  Landscape of helper and regulatory antitumor CD4+ T cells in mela…
    L0 2021 cites=  127 abs=y  Single cell characterization of the immune microenvironment of me…
2) forward-citation layers
    layer 1: 783 candidates citing 5 parents -> kept 8 of top, pool 775 (years 2023..2026)
    layer 2: 176 candidates citing 8 parents -> kept 8 of top, pool 167 (years 2024..2026)
    layer 3: 135 candidates citing 8 parents -> kept 8 of top, pool 127 (years 2024..2026)
3) top entry: latest by author / by topic
    latest[author] 2026 cites=  4  A longitudinal, multi-omic atlas reveals the emergence of a s…
    latest[author] 2026 cites=  3  RB loss modulates chro

In [12]:
def quality_color(q):
    if q is None:
        return T["unknown"]
    return T["quality"][int(round(min(max(q, 0.0), 1.0) * 6))]


def tooltip(rec):
    esc = _html.escape
    authors = ", ".join(n for _, n in rec["authors"][:3]) + (" et al." if len(rec["authors"]) > 3 else "")
    q = "n/a" if rec["quality"] is None else f"{rec['quality']:.2f}"
    ppy = f"{rec['ppy']:,}/yr" if rec["ppy"] else "unknown"
    bits = [
        f"<b>{esc(rec['title'])}</b>",
        f"<span style='color:{T['ink2']}'>{esc(authors)}</span>",
        f"{esc(rec['source_name'] or rec['venue'] or 'no venue')} · {rec['year']} · {rec['cites']} citations · layer {rec['layer']}",
    ]
    if rec["retracted"]:
        bits.append(f"<b style='color:{T['retracted']}'>⚠ retracted publication</b>")
    bits.append(f"venue quality {q} — journal volume {ppy}")
    bits += [f"• {esc(ev)}" for ev in rec["evidence"]]
    if rec["explanation"]:
        bits.append(f"<i>{esc(rec['explanation'])[:420]}</i>")
    links = []
    if rec["doi"]:
        links.append(f"<a href='https://doi.org/{rec['doi']}'>DOI</a>")
    links.append(f"<a href='https://www.semanticscholar.org/paper/{rec['id']}'>Semantic&nbsp;Scholar</a>")
    if rec["pmid"]:
        links.append(f"<a href='https://pubmed.ncbi.nlm.nih.gov/{rec['pmid']}/'>PubMed</a>")
    bits.append(" · ".join(links))
    return (f"<div style='font:12.5px/1.55 system-ui,sans-serif;max-width:400px;color:{T['ink']}'>"
            + "<br>".join(bits) + "</div>")


def legend_html():
    ramp = ",".join(T["quality"])

    def chip(color, dash=None, w=3):
        d = f" stroke-dasharray='{dash}'" if dash else ""
        return (f"<svg width='44' height='10' style='vertical-align:middle'>"
                f"<line x1='2' y1='5' x2='42' y2='5' stroke='{color}' stroke-width='{w}'{d}/></svg>")

    def dot(color, border=None):
        b = f";border:2px solid {border}" if border else ""
        return f"<span style='display:inline-block;width:11px;height:11px;border-radius:50%;" \
               f"background:{color}{b};vertical-align:middle'></span>"

    return f"""
    <div style="position:fixed;top:14px;right:14px;z-index:999;pointer-events:none;
                background:{T['surface']}ee;border:1px solid {T['grid']};border-radius:10px;
                padding:10px 14px;font:12px/1.6 system-ui,sans-serif;color:{T['ink']};max-width:330px">
      <b>How to read this graph</b><br>
      <b>Y axis = publication year</b> — newest at the top, seeds anchor the bottom<br>
      {dot(T['seed'], T['ink'])} seed paper (constant orange) ·
      {dot(T['quality'][5])} discovered · {dot(T['quality'][0])} low quality ·
      {dot(T['unknown'])} unknown venue · ⚠ border = retracted<br>
      <span style="background:linear-gradient(90deg,{ramp});display:inline-block;width:130px;height:10px;
                   border-radius:2px;vertical-align:middle"></span>&nbsp; venue quality: low → high
      <span style="color:{T['ink2']}">(exact number in tooltip & table)</span><br>
      {chip(T['citation'])} <b>citation</b> — arrow points at what it cites<br>
      {chip(T['semantic'], '7,5', 2.5)} <b>semantic</b> — abstract similarity<br>
      {chip(T['author'], '2,5', 2.5)} <b>author</b> — shared author<br>
      edge width = strength (hover an edge for the reason + exact number)<br>
      <b>click</b> a node to copy its DOI link · <b>double-click</b> a node to load
      more candidates citing it (gray = not yet assessed)<br>
      <span style="color:{T['ink2']}">quality = journal papers/year heuristic
      (paper mills publish thousands/yr); not a verdict on the paper</span>
    </div>"""


CLICK_JS = """
    network.on('click', function (params) {
        if (params.nodes.length > 0) {
            var n = nodes.get(params.nodes[0]);
            if (n && n.url) { copyText(n.url); }
        }
    });
    // dynamic loading in the UI: double-click a paper to reveal additional
    // candidates from its layer pool (pre-loaded as hidden nodes/edges)
    network.on('doubleClick', function (params) {
        if (params.nodes.length > 0) {
            var id = params.nodes[0];
            var inc = edges.get({ filter: function (e) { return e.hidden === true && (e.from === id || e.to === id); } });
            if (inc.length === 0) { toast('no more candidates pooled for this paper'); return; }
            var edgeUpd = inc.map(function (e) { return { id: e.id, hidden: false }; });
            var show = {};
            inc.forEach(function (e) { show[e.from === id ? e.to : e.from] = true; });
            var nodeUpd = Object.keys(show).map(function (nid) { return { id: nid, hidden: false }; });
            edges.update(edgeUpd);
            nodes.update(nodeUpd);
            toast('+' + inc.length + ' connection' + (inc.length > 1 ? 's' : '') + ' loaded');
        }
    });
    var toastDiv = document.createElement('div');
    toastDiv.style.cssText = 'position:fixed;bottom:18px;left:50%;transform:translateX(-50%);' +
        'background:%INK%;color:%SURF%;padding:8px 14px;border-radius:8px;font:13px system-ui,' +
        'sans-serif;display:none;z-index:9999;max-width:70%;overflow:hidden;text-overflow:ellipsis;white-space:nowrap';
    document.body.appendChild(toastDiv);
    function toast(msg) {
        toastDiv.textContent = msg;
        toastDiv.style.display = 'block';
        setTimeout(function () { toastDiv.style.display = 'none'; }, 1800);
    }
    function fallbackCopy(txt) {
        var ta = document.createElement('textarea');
        ta.value = txt; document.body.appendChild(ta); ta.select();
        try { document.execCommand('copy'); toast('Copied: ' + txt); }
        catch (e) { toast('Link: ' + txt); }
        document.body.removeChild(ta);
    }
    function copyText(txt) {
        if (navigator.clipboard && navigator.clipboard.writeText) {
            navigator.clipboard.writeText(txt).then(
                function () { toast('Copied: ' + txt); },
                function () { fallbackCopy(txt); });
        } else { fallbackCopy(txt); }
    }
    return network;
"""


def edge_tooltip(e):
    a, b = PAPERS[e["src"]], PAPERS[e["dst"]]
    ta, tb = short_title(a["title"], 46), short_title(b["title"], 46)
    strength = min(float(e["strength"] or 0), 1.0)
    if e["type"] == "citation":
        body = f"'{ta}' <b>cites</b> '{tb}' — verified citation link (Semantic Scholar)"
    elif e["type"] == "semantic":
        body = f"<b>semantic</b> — abstract similarity ({e.get('detail', '')}) between '{ta}' and '{tb}'"
    else:
        body = f"<b>shared author: {e['label']}</b> ({e.get('detail', '')}) on '{ta}' and '{tb}'"
    return (f"<div style='font:12.5px/1.5 system-ui,sans-serif;max-width:340px;color:{T['ink']}'>"
            f"{body}<br>strength {strength:.2f} of 1.00 → edge width</div>")


def render():
    """Y axis = publication year (newest on top); X = per-layer spring layout."""
    net = Network(height="1020px", width="100%", directed=True, bgcolor=T["surface"],
                  font_color=T["ink"], cdn_resources="in_line")
    net.toggle_physics(False)
    # --- vertical positions from publication year
    years = [r["year"] for r in PAPERS.values() if r["year"]]
    y_min, y_max = (min(years), max(years)) if years else (2016, 2026)
    YS = 92  # px per year
    layer_median_year = {}
    by_layer = defaultdict(list)
    for pid, r in PAPERS.items():
        by_layer[r["layer"] or 0].append(pid)
    for layer, pids in by_layer.items():
        ys = [PAPERS[p]["year"] for p in pids if PAPERS[p]["year"]]
        layer_median_year[layer] = sorted(ys)[len(ys) // 2] if ys else y_min

    def y_of(rec):
        year = rec["year"] or layer_median_year.get(rec["layer"] or 0, y_min)
        jitter = (int(rec["id"][:6], 16) % 22) - 11      # de-stack same-year nodes
        return -(year - y_min) * YS + jitter

    # --- horizontal positions: per-layer spring layout, layers offset to fill
    # the canvas instead of stacking every cluster on the same x band
    LAYER_X = {0: -240, 1: 40, 2: 280, 3: -60}
    scores = [r["score"] or 0 for r in PAPERS.values()] or [0]
    smin, smax = min(scores), (max(scores) or 1)
    POS = {}  # remember real-node positions for placing hidden candidates nearby
    for layer, pids in sorted(by_layer.items()):
        G = nx.Graph()
        G.add_nodes_from(pids)
        for e in EDGES:
            if e["src"] in pids and e["dst"] in pids:
                G.add_edge(e["src"], e["dst"])
        if len(pids) == 1:
            xs = {pids[0]: 0.0}
        else:
            xy = nx.spring_layout(G, k=1.3, seed=7)
            lo, hi = min(v[0] for v in xy.values()), max(v[0] for v in xy.values())
            span, spread = (hi - lo) or 1.0, 150 * min(len(pids), 13)
            xs = {p: (xy[p][0] - (lo + hi) / 2) / span * spread for p in pids}
        x_off = LAYER_X.get(layer, 0)
        for p in pids:
            rec = PAPERS[p]
            jitter = (int(p[:4], 16) % 60) - 30          # deterministic de-overlap
            link = f"https://doi.org/{rec['doi']}" if rec["doi"] else \
                   f"https://www.semanticscholar.org/paper/{rec['id']}"
            border = T["retracted"] if rec["retracted"] else (T["ink"] if rec["is_seed"] else T["grid"])
            size = 15 if rec["is_seed"] else 9 + 9 * (((rec["score"] or 0) - smin) / (smax - smin or 1))
            net.add_node(p,
                         label=("⚠ " if rec["retracted"] else "") + short_title(rec["title"], 26),
                         shape="dot", x=xs[p] + jitter + x_off, y=y_of(rec),
                         size=size, borderWidth=2.5 if rec["is_seed"] else (2 if rec["retracted"] else 1),
                         color={"border": border,
                                "background": T["seed"] if rec["is_seed"] else quality_color(rec["quality"]),
                                "highlight": {"border": T["citation"]}},
                         font={"color": T["ink"], "size": 11, "face": "system-ui, sans-serif"},
                         title=tooltip(rec), url=link)
            POS[p] = (xs[p] + jitter + x_off, y_of(rec))
    # --- year axis ticks (ghost text nodes on the left margin)
    step = 2 if (y_max - y_min) > 12 else 1
    for year in range(y_min, y_max + 1, step):
        net.add_node(f"__year_{year}", label=str(year), shape="text", x=-1350,
                     y=-(year - y_min) * YS, physics=False,
                     color=T["muted"], font={"color": T["muted"], "size": 13})
    # --- hidden pool candidates: pre-loaded for double-click reveal.
    # Gray = not yet quality-assessed; use more()/inspect() for full treatment.
    by_parent = defaultdict(list)
    for _layer, pool in LAYER_POOLS.items():
        for s, cid, plist, meta in pool:
            for p in plist:
                if p in PAPERS:
                    by_parent[p].append((s, cid, meta))
    revealed_cap, hidden_added = 6, set()
    for pid, entries in sorted(by_parent.items(), key=lambda kv: -len(kv[1])):
        if pid not in POS:
            continue
        entries.sort(key=lambda x: -x[0])
        px, py = POS[pid]
        parent = PAPERS[pid]
        for s, cid, meta in entries[:revealed_cap]:
            if cid in hidden_added or cid in PAPERS:
                continue
            hidden_added.add(cid)
            year = meta.get("year") or parent["year"] or y_max
            tip = ("<div style='font:12.5px/1.55 system-ui,sans-serif;max-width:360px;"
                   f"color:{T['ink']}'><b>{_html.escape((meta.get('title') or '')[:80])}</b><br>"
                   f"{_html.escape(meta.get('venue') or '')} · {year} · pool candidate "
                   f"(relevance {s:.1f})<br><span style='color:{T['ink2']}'>double-click revealed; "
                   "quality not yet assessed — run more() in the notebook to integrate fully</span></div>")
            net.add_node(cid, label=short_title(meta.get("title", ""), 26), shape="dot",
                         x=px + ((int(cid[:4], 16) % 300) - 150),
                         y=-(year - y_min) * YS + ((int(cid[5:9], 16) % 22) - 11),
                         size=7, borderWidth=1, hidden=True,
                         color={"border": T["grid"], "background": T["unknown"]},
                         font={"color": T["ink2"], "size": 10, "face": "system-ui, sans-serif"},
                         title=tip)
            net.add_edge(cid, pid, color=T["citation"], width=1.3, arrows="to", hidden=True,
                         title=("<div style='font:12.5px/1.5 system-ui,sans-serif;max-width:340px;"
                                f"color:{T['ink']}'>pool candidate: '{short_title(meta.get('title', ''), 40)}' "
                                f"<b>cites</b> '{short_title(parent['title'], 40)}' (relevance {s:.1f})</div>"))
    # --- edges (width 0.8..5.0 so strength differences are visible)
    cite_out = Counter(e["src"] for e in EDGES if e["type"] == "citation")
    for e in EDGES:
        t = e["type"]
        if t == "citation":
            # a paper woven into the corpus (citing several graph nodes) gets
            # stronger edges than a one-off citers
            strength = 0.35 + 0.15 * min(cite_out.get(e["src"], 1), 4)
        else:
            strength = min(float(e["strength"] or 0), 1.0)
        kw = dict(
            color=T[t],
            width=0.8 + 4.2 * strength,
            dashes=False if t == "citation" else [7, 5] if t == "semantic" else [2, 5],
            smooth={"enabled": True, "type": "curvedCW",
                    "roundness": {"citation": 0.0, "semantic": 0.18, "author": -0.18}[t]},
            title=edge_tooltip(e),
        )
        # pyvis auto-adds arrows on directed graphs; override for non-citation edges
        kw["arrows"] = "to" if t == "citation" else ""
        net.add_edge(e["src"], e["dst"], **kw)
    net.set_options(json.dumps({
        "interaction": {"hover": True, "tooltipDelay": 60, "hoverConnectedEdges": True,
                        "selectConnectedEdges": False},
        "edges": {"selectionWidth": 1.2, "hoverWidth": 1.6},
        "nodes": {"borderWidthSelected": 3},
        "physics": {"enabled": False},
    }))
    net.write_html(GRAPH_HTML, open_browser=False)
    page = Path(GRAPH_HTML).read_text()
    page = page.replace("return network;",
                        CLICK_JS.replace("%INK%", T["ink"]).replace("%SURF%", T["surface"]), 1)
    # pyvis's hover popup only looks up nodes — hovering an edge throws and shows
    # nothing; fall back to the edge's own title
    page = page.replace("popup.innerHTML = nodeData[0].title;",
                        "popup.innerHTML = (nodeData[0] && nodeData[0].title) ||"
                        " (edges.get(nodeId) || {}).title || '';")
    page = page.replace("</body>", legend_html() + "</body>")
    Path(GRAPH_HTML).write_text(page)
    display(IFrame(GRAPH_HTML, width="100%", height="1040"))


def results_table():
    """The table view: same information as the graph, sortable, linkable."""
    rows = []
    for pid, r in sorted(PAPERS.items(), key=lambda kv: (-(kv[1]["layer"] or 0), -(kv[1]["score"] or 0))):
        where = "SEED" if r["is_seed"] else f"L{r['layer']}" + ("·latest" if r["entry"] else "")
        rows.append({
            "id": short(pid),
            "where": where,
            "year": r["year"],
            "score": r["score"],
            "quality": None if r["quality"] is None else round(r["quality"], 2),
            "sem": r["sem"],
            "journal ppy": r["ppy"],
            "venue": (r["source_name"] or r["venue"] or "")[:26],
            "title": short_title(r["title"], 56),
            "why (evidence)": " | ".join(r["evidence"])[:110],
        })
    df = pd.DataFrame(rows)
    display(df)
    return df


print("rendering functions ready")
render()
results_table();

rendering functions ready


,id,where,year,score,quality,sem,journal ppy,venue,title,why (evidence)
0,e677454a,L3·latest,2026,23.0,0.72,0.091,545,JCI Insight,Loss of tumor-infiltrating lymphocytes and poo...,"shares author(s): M. Sade-Feldman, M. Sade-Fel..."
1,2556e00b,L3·latest,2026,23.0,NaN,0.087,10946,medRxiv,Immune and stromal features of durable complet...,abstract similarity cosine 0.53 (tfidf) with A...
2,174e78ba,L3·latest,2026,21.0,0.19,0.070,8266,Cancer Research,Abstract B096: Immune and stromal features of ...,abstract similarity cosine 0.53 (tfidf) with I...
3,4cc695ce,L3,2026,20.0,1.00,0.057,195,Immunology,Barriers and Strategies to Enhance CAR‐T Cell ...,cites 1 discovered paper(s) on paths to the se...
4,65c3bd7e,L3,2026,19.0,1.00,0.042,165,Med,Current state of CAR-T cell therapies for soli...,cites 1 discovered paper(s) on paths to the se...
5,c188d3fb,L3,2026,19.0,0.73,0.040,533,MedComm,Targeted Therapies for Each Subtype of Breast ...,cites 1 discovered paper(s) on paths to the seeds
6,96829c4a,L3·latest,2026,19.0,0.77,0.073,455,Cell Reports Medicine,"A longitudinal, multi-omic atlas reveals the e...",shares author(s): M. Herlyn | latest work (ent...
7,ee9e8f75,L3,2025,18.0,0.86,0.028,332,Molecular Cancer,Cellular senescence in cancer: from mechanism ...,cites 1 discovered paper(s) on paths to the seeds
8,ee050555,L3,2026,18.0,0.69,0.030,598,Cancer Letters,Emerging biomarkers in melanoma: Bridging mole...,cites 1 discovered paper(s) on paths to the seeds
9,e4470690,L3,2025,18.0,0.73,0.034,533,MedComm,Toll‐Like Receptors in the Immunotherapy Era: ...,cites 1 discovered paper(s) on paths to the seeds


## Explore: double-click · more · inspect · expand · reset

**Dynamic loading, two levels:**

- **In the graph (instant): double-click any paper node** — additional candidates that cite it slide in, pre-loaded from that layer's scored pool. They appear **gray** because their quality and similarity haven't been assessed yet; hover them for the pool score, and use the notebook calls below to integrate them fully.
- `more()` — the full version: promote the next candidates (`more(2)` for layer 2 only, `more([1, 2], k=6)` for several at once) with quality scores, semantic/author edges, and explanations computed.

**The rest of the loop:**

- `inspect("8-char-id")` — everything about one paper (metadata, quality breakdown, evidence, explanation, abstract, links). Copy an `id` from the table above.
- `expand("8-char-id")` — promote a paper to an extra seed and **re-grow the graph around it** (cached API calls keep this fast).
- `reset_explore()` — back to the original seeds.

Also in the graph itself: hover a node for its full card, hover an edge for the connection reason and its strength, click a node to copy its DOI link.

In [13]:
def _resolve(pid_prefix):
    m = [p for p in PAPERS if p.startswith(pid_prefix)]
    if not m:
        print(f"no paper id starts with {pid_prefix!r} — copy one from the table's 'id' column")
    elif len(m) > 1:
        print(f"ambiguous prefix {pid_prefix!r}: {[short(x) for x in m]}")
    return m[0] if len(m) == 1 else None


def inspect(pid_prefix):
    pid = _resolve(pid_prefix)
    if not pid:
        return None
    r = PAPERS[pid]
    print("=" * 96)
    print(f"{r['title']}  ({r['year']})")
    print("authors: " + ", ".join(n for _, n in r["authors"][:8]) + (" …" if len(r["authors"]) > 8 else ""))
    print(f"venue: {r['source_name'] or r['venue'] or 'unknown'} | layer {r['layer']} | "
          f"score {r['score']} | quality {round(r['quality'], 2) if r['quality'] is not None else '?'} "
          f"(journal {format(r['ppy'], ',') if r['ppy'] else '?'} papers/yr)")
    for ev in r["evidence"]:
        print("  •", ev)
    if r["explanation"]:
        print("  relevance:", r["explanation"])
    if r["abstract"]:
        print("  abstract:", r["abstract"][:700].replace("\n", " "), "…")
    link = f"https://doi.org/{r['doi']}" if r["doi"] else f"https://www.semanticscholar.org/paper/{r['id']}"
    print("  link:", link)
    return r


def expand(pid_prefix):
    pid = _resolve(pid_prefix)
    if not pid:
        return
    r = PAPERS[pid]
    if not r["doi"]:
        print("no DOI for this paper — cannot expand")
        return
    if r["doi"] not in EXTRA_SEEDS:
        EXTRA_SEEDS.append(r["doi"])
    print(f"re-growing the graph around: {short_title(r['title'], 70)}\n")
    run_pipeline()
    render()
    results_table()


def more(layers=None, k=4):
    """Dynamically load more candidates into layer(s) from the unused pool.

    more()      -> next 4 candidates in every layer
    more(2)     -> next 4 in layer 2          more([1, 2], k=6) -> 6 each
    """
    wanted = sorted(LAYER_POOLS) if layers is None else ([layers] if isinstance(layers, int) else list(layers))
    n_added = 0
    for layer in wanted:
        pool = LAYER_POOLS.get(layer) or []
        taken = 0
        while pool and taken < k:
            _s, cid, plist, meta = pool.pop(0)
            if cid in PAPERS:
                continue
            if add_paper(meta, layer=layer):
                for p in plist:
                    add_edge(cid, p, "citation", 0.55)
                taken += 1
                n_added += 1
                print(f"    +L{layer} {meta.get('year')} {short_title(meta.get('title', ''), 60)}")
    if not n_added:
        print("pools exhausted for those layers — run_pipeline() again after expand() to refill")
        return
    enrich_quality()
    build_semantic()
    build_author_edges()
    finalize()
    explain_all()
    print(f"added {n_added} papers -> {len(PAPERS)} nodes")
    render()
    results_table()


def reset_explore():
    EXTRA_SEEDS.clear()
    print("back to the original seeds\n")
    run_pipeline()
    render()
    results_table()


top = max((r for r in PAPERS.values() if not r["is_seed"]), key=lambda r: r["score"] or 0)
print(f"Highest-scoring discovered paper ({short(top['id'])}) — try expand('{short(top['id'])}') next:\n")
inspect(top["id"])

Highest-scoring discovered paper (2749e634) — try expand('2749e634') next:

CD39 on CD8+ Tumour‐Infiltrating Lymphocytes: A Paradoxical Marker of Tumour Reactivity and Immunosuppression  (2026)
authors: M. Sambucci, D. Fenoglio, F. Ferrera, Laura Camporesi, Andrea Lagazio, G. Borsellino, L. Battistini, G. Filaci
venue: European Journal of Immunology | layer 1 | score 47 | quality 0.93 (journal 261 papers/yr)
  • cites 3 seed paper(s): Dissecting the multicellular ecosystem …; Defining T cell states associated with …; Phenotype, specificity and avidity of a…
  • cites 1 discovered paper(s) on paths to the seeds
  relevance: cites 3 seed paper(s): Dissecting the multicellular ecosystem …; Defining T cell states associated with …; Phenotype, specificity and avidity of a… | cites 1 discovered paper(s) on paths to the seeds
  abstract: ABSTRACT CD39 (ENTPD1) occupies a paradoxical position in the immunology of solid tumours. On CD8+ tumour‐infiltrating lymphocytes (TILs), its surface expres

{'id': '2749e634292c917319de11a3196554af02e7e8c3',
 'title': 'CD39 on CD8+ Tumour‐Infiltrating Lymphocytes: A Paradoxical Marker of Tumour Reactivity and Immunosuppression',
 'year': 2026,
 'venue': 'European Journal of Immunology',
 'abstract': 'ABSTRACT CD39 (ENTPD1) occupies a paradoxical position in the immunology of solid tumours. On CD8+ tumour‐infiltrating lymphocytes (TILs), its surface expression serves as one of the most reliable available markers of genuine tumour‐antigen experience and identifies the clonally expanded tissue‐resident and cytolytically competent T cells, that are largely absent from the bystander populations, which are specific for tumour‐unrelated antigens and numerically dominate many tumour infiltrates. Yet the same enzymatic activity that defines CD39, the hydrolysis of extracellular ATP (eATP) to AMP, directly feeds CD73‐dependent adenosine generation. These very cells therefore actively perpetuate the immunosuppressive purinergic milieu that constrains

## Notes, limits, next steps

**Honest limits** (good demo talking points)

- The quality color is a **venue heuristic** — journal papers/year plus a retraction flag — not a judgment of a paper. Two guards keep it sane: preprint servers (bioRxiv & co.) are *gray* (not peer-review-graded venues), and flagship weeklies (venue h-index ≥ 900: Nature, Science, Cell…) are floored at 0.75 despite their volume. Every tooltip still shows the raw papers/year number.
- Candidate windows are **recency-sliced**: Semantic Scholar lists citers newest-first, and layer 1 additionally samples an older "established followers" window — very popular older followers of a seed can still be missed. That is a deliberate frontier bias, matching the "latest on top" idea.
- Semantic edges are abstract/title similarity; the TF-IDF fallback is coarser than embedding-based similarity (set a Token Factory key and an `EMBED_MODEL` to switch).
- Author edges depend on Semantic Scholar author disambiguation.
- The newest papers have no citations yet — that is exactly why the top layer also enters by *author* and by *topic*.

**Toward the hackathon app (Nebius)**

- The pipeline functions are the backend: seeds in → JSON graph out; Token Factory supplies embeddings + explanations (required by the brief); the pyvis HTML becomes a thin web shell.
- API budget at demo time: one fresh build ≈ 40 Semantic Scholar calls + ~80 free OpenAlex record calls, all cached afterwards. A free Semantic Scholar API key raises the rate ceiling; OpenAlex query endpoints (unused here) would also become attractive from a non-shared IP.
- Experiments worth running with a real researcher: does layer position beat plain recency for finding the next paper to read? Do the explanation tooltips change their picks?